# NewsQA RAG — chunking strategy study (Kaggle)

Compares **hierarchical** against **recursive** chunking at the same chunk
size, and reports retrieval at **@3, @5 and @7**.

Hierarchical splits each article into parents, then splits each parent into
children; the children are what gets embedded and retrieved, so a query
matches a tight passage instead of being diluted across a long one. Each
child keeps its `parent_id`, so a later stage can widen a hit back out to
the parent for the generator to read.

Same harness as notebook 08 — same 50 development articles, same BGE-M3
sparse index, same rerankers — with strategy added as a third dimension.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='62d7e200ca685963e64931863eb2e9a4b047eb16'
HF_REPO_ID='MatchaMacchiato/newsqa_200_11064_v2.0.0'
HF_REVISION='b81c8db6847a23272665946c0c43c72e9a212fd9'  # the v2.0.0 commit; swap for 'v2.0.0' once that tag exists
# (chunk_size, chunk_overlap, strategy). Recursive 512/64 is the control: it
# repeats notebook 08's middle variant, so any difference here is the strategy
# and not the size. Hierarchical children default to half the parent size, so
# 512/64 indexes ~256-token children and 1024/128 indexes ~512-token ones.
CHUNK_VARIANTS=[(512,64,'recursive'),(512,64,'hierarchical'),(1024,128,'hierarchical')]
# Scoring only reports cut-offs up to rerank_top_n, so @7 needs at least 7
# survivors. Costs no extra GPU: the cross-encoder scores all top_k first.
RERANK_TOP_N=7
RERANKER_MODELS=['cross-encoder/ms-marco-MiniLM-L-6-v2','BAAI/bge-reranker-large']
DEVELOPMENT_ARTICLES=50
SMOKE_QUESTIONS=None  # Set to 5 only for a pipeline smoke test.
CLEANUP_COMPLETED_VARIANTS=True
AUTO_RESTORE_FROM_INPUT=True
MIN_FREE_GIB=5
KAGGLE_WORKING=Path('/kaggle/working')
KAGGLE_INPUT=Path('/kaggle/input')
PROJECT_ROOT=KAGGLE_WORKING/'Text-Mining---NewsQA-RAG'
WORK_ROOT=KAGGLE_WORKING/'newsqa_chunk_strategy'
DATA_ROOT=WORK_ROOT/'data'
INDEX_ROOT=WORK_ROOT/'indexes'
EXPERIMENTS=WORK_ROOT/'experiments'
SPECS=WORK_ROOT/'specs'
CACHE=WORK_ROOT/'retrieval_cache'
RESULTS=WORK_ROOT/'results'
LOGS=WORK_ROOT/'logs'
STATE_MARKER=WORK_ROOT/'newsqa_chunk_strategy_state.json'


## 1. Kaggle setup
Enable Internet and a GPU accelerator. Add the private-dataset token as the Kaggle secret `HF_TOKEN`. To resume, attach the output of an earlier version of this notebook as an input.

In [ ]:
# The dataset is public, so a token is optional. Set one as the Kaggle
# secret HF_TOKEN only to lift anonymous download rate limits.
token=''
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception:
    pass
if token:
    os.environ['HF_TOKEN']=token
else:
    print('No HF_TOKEN secret; downloading the public dataset anonymously.')
os.environ['HF_HOME']=str(KAGGLE_WORKING/'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0'})
if AUTO_RESTORE_FROM_INPUT and not WORK_ROOT.exists() and KAGGLE_INPUT.exists():
    markers=sorted(KAGGLE_INPUT.rglob('newsqa_round3_state.json'),key=lambda p:p.stat().st_mtime,reverse=True)
    if markers:
        previous=markers[0].parent
        WORK_ROOT.mkdir(parents=True,exist_ok=True)
        for name in ['experiments','results','specs','retrieval_cache']:
            source=previous/name
            if source.exists(): shutil.copytree(source,WORK_ROOT/name,dirs_exist_ok=True)
        shutil.copy2(markers[0],STATE_MARKER)
        print('Restored compact Round 3 state from:',previous,flush=True)
for path in [DATA_ROOT,INDEX_ROOT,EXPERIMENTS,SPECS,CACHE,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--depth','30',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import torch, yaml, pandas as pd
from IPython.display import display, Image
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print('GPU:',torch.cuda.get_device_name(0),round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GiB')
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())


## 2. Experimental protocol
Each chunk corpus is built and evaluated sequentially against the complete 11,064-article corpus. The same BGE-M3 index serves both rerankers and both question variants. Selection uses only resolved development questions; the held-out final-test partition is not touched.

In [ ]:
def disk_status():
    usage=shutil.disk_usage(KAGGLE_WORKING)
    value={'free_gib':round(usage.free/2**30,2),'used_gib':round(usage.used/2**30,2),'total_gib':round(usage.total/2**30,2)}
    print('Disk:',value,flush=True); return value
def require_disk():
    status=disk_status()
    if status['free_gib']<MIN_FREE_GIB: raise RuntimeError(f"Only {status['free_gib']} GiB free; clean Kaggle working storage before continuing")
def run_command(command,label):
    log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(map(str,command)),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen([str(v) for v in command],cwd=PROJECT_ROOT,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); log.flush()
        returncode=process.wait()
    if returncode: raise subprocess.CalledProcessError(returncode,command)
    return log_path
def write_spec(variant_id,index_item,variant_root):
    # Winner selection stays on @5 so this stays comparable with 08.
    profile=f'chunk_{variant_id}'
    testsets={'resolved':str(variant_root/'final_deduplicated/testset_resolved.jsonl'),'original':str(variant_root/'final_deduplicated/testset_reviewed_original.jsonl')}
    runs=[]
    for question_variant in ['resolved','original']:
        for model in RERANKER_MODELS:
            runs.append({'index':profile,'variant':question_variant,'retriever':'sparse','reranker':'cross-encoder','reranker_model':model})
    spec={'schema_version':1,'experiment':{'id':f'phase1-round3-{variant_id.replace("_","-")}','name':f'Phase 1 Round 3 {variant_id}'},'output_dir':str(EXPERIMENTS),'seed':42,'dataset':{'article_field':'article_key','development_articles':DEVELOPMENT_ARTICLES,'indexes':{profile:{'config':index_item['config_path'],'variant_manifest':index_item['variant_manifest'],'testsets':testsets}}},'fixed':{'retrieval_only':True,'top_k':20,'rerank_top_n':RERANK_TOP_N,'partition':'development'},'runs':runs,'runtime':{'max_attempts':2,'progress':True,'shared_retrieval_cache':str(CACHE),**({'n_eval':SMOKE_QUESTIONS} if SMOKE_QUESTIONS else {})},'judge':{'enabled':False},'summary':{'metrics':['retrieval.hit_rate@1','retrieval.hit_rate@3','retrieval.mrr@3','retrieval.ndcg@3','retrieval.hit_rate@5','retrieval.mrr@5','retrieval.ndcg@5','retrieval.recall@5','retrieval.hit_rate@7','retrieval.mrr@7','retrieval.ndcg@7','retrieval.recall@7'],'paired_metric':'retrieval.mrr@5','quality_metric':'retrieval.mrr@5.mean','latency_metric':'latency.total.p50_ms'}}
    path=SPECS/f'round3_{variant_id}.yaml'; path.write_text(yaml.safe_dump(spec,sort_keys=False),encoding='utf-8'); return path,spec['experiment']['id']
def update_state(completed):
    STATE_MARKER.write_text(json.dumps({'schema_version':1,'updated_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'completed_variants':sorted(completed),'repo_commit':REPO_COMMIT},indent=2)+'\n',encoding='utf-8')
def cleanup_variant(variant_root,index_dir):
    if not CLEANUP_COMPLETED_VARIANTS: return
    for path in [variant_root,index_dir]:
        if path.exists(): shutil.rmtree(path)
    print('Removed completed corpus and index:',variant_root.name,flush=True); disk_status()


## 3. Run the Round 3 matrix
The order prioritizes resolved questions and MiniLM first. Completed configurations are recognized by their copied `comparison.json` and skipped on resume.

In [ ]:
completed=set(json.loads(STATE_MARKER.read_text()).get('completed_variants',[])) if STATE_MARKER.exists() else set()
for size,overlap,strategy in CHUNK_VARIANTS:
    variant_id=f'{size}_{overlap}_{strategy}'
    result_comparison=RESULTS/'comparisons'/f'{variant_id}.json'
    if result_comparison.exists():
        completed.add(variant_id); print('Skipping completed variant:',variant_id,flush=True); continue
    require_disk()
    variant_name=f'chunk_{size}_{overlap}_{strategy}'
    variant_root=DATA_ROOT/variant_name
    index_dir=INDEX_ROOT/variant_id
    run_command([sys.executable,'-u','scripts/build_ablation_datasets.py','--repo-id',HF_REPO_ID,'--revision',HF_REVISION,'--output-base',DATA_ROOT,'--chunk-sizes',size,'--chunk-overlaps',overlap,'--strategies',strategy,'--db-path-base',DATA_ROOT/'temporary_db','--skip-vector-index'],f'materialize_{variant_id}')
    index_manifest=index_dir/'index_manifest.json'
    if not index_manifest.exists():
        run_command([sys.executable,'-u','scripts/build_retrieval_models_index.py','--chunks-path',variant_root/'final_deduplicated/chunks.jsonl','--base-variant-manifest',variant_root/'manifests/deduplicated.variant.json','--output-dir',index_dir,'--sparse-ids','bge_m3_sparse','--skip-dense','--device','cuda'],f'index_{variant_id}')
    manifest=json.loads(index_manifest.read_text()); item=manifest['sparse_indexes']['bge_m3_sparse']
    spec_path,experiment_id=write_spec(variant_id,item,variant_root)
    run_command([sys.executable,'-u','scripts/run_experiment.py',spec_path],f'experiment_{variant_id}')
    experiment_dir=EXPERIMENTS/experiment_id
    run_command([sys.executable,'-u','scripts/summarize_experiments.py',experiment_dir],f'summarize_{variant_id}')
    (RESULTS/'comparisons').mkdir(parents=True,exist_ok=True)
    shutil.copy2(experiment_dir/'comparison.json',result_comparison)
    shutil.copy2(spec_path,RESULTS/f'spec_{variant_id}.yaml')
    completed.add(variant_id); update_state(completed); cleanup_variant(variant_root,index_dir)
print('Completed variants:',sorted(completed))


## 4. Aggregate results and select using resolved questions

In [ ]:
sys.path.insert(0,str(PROJECT_ROOT/'backend'))
from newsqa_rag.evaluation.phase1 import load_comparison_rows, select_winner, write_rows_csv
comparison_paths=sorted((RESULTS/'comparisons').glob('*.json'))
rows=load_comparison_rows(comparison_paths)
write_rows_csv(rows,RESULTS/'round3.csv')
resolved=pd.DataFrame([row for row in rows if row.get('variant')=='resolved'])
original=pd.DataFrame([row for row in rows if row.get('variant')=='original'])
columns=['index','reranker_model','retrieval.hit_rate@5.mean','retrieval.mrr@5.mean','retrieval.ndcg@5.mean','latency.total.p50_ms','coverage.success_rate']
display(resolved[columns].sort_values('retrieval.mrr@5.mean',ascending=False))
winner=select_winner(rows,variant='resolved')
(RESULTS/'round3_resolved_winner.json').write_text(json.dumps(winner,indent=2,sort_keys=True)+'\n')
print('Primary resolved winner:'); display(winner)
paired=pd.concat([resolved.assign(question_variant='resolved'),original.assign(question_variant='original')]).pivot_table(index=['index','reranker_model'],columns='question_variant',values=['retrieval.hit_rate@5.mean','retrieval.mrr@5.mean','retrieval.ndcg@5.mean','latency.total.p50_ms']).reset_index()
display(paired)


## 5. Resolved accuracy-latency visualization and exports

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
figures=RESULTS/'figures'; figures.mkdir(exist_ok=True)
plot=resolved.copy(); plot['reranker']=plot['reranker_model'].map({'cross-encoder/ms-marco-MiniLM-L-6-v2':'MiniLM','BAAI/bge-reranker-large':'BGE-large'})
plt.figure(figsize=(8,5)); sns.scatterplot(data=plot,x='latency.total.p50_ms',y='retrieval.hit_rate@5.mean',hue='reranker',style='index',s=130)
plt.xlabel('Median latency (ms)'); plt.ylabel('Resolved Hit Rate@5'); plt.tight_layout(); plt.savefig(figures/'resolved_quality_latency.png',dpi=200); plt.show(); plt.close()
plt.figure(figsize=(9,5)); sns.barplot(data=plot,x='index',y='retrieval.mrr@5.mean',hue='reranker')
plt.xlabel('Chunk configuration'); plt.ylabel('Resolved MRR@5'); plt.xticks(rotation=15); plt.tight_layout(); plt.savefig(figures/'resolved_mrr_by_chunking.png',dpi=200); plt.show(); plt.close()
archive=shutil.make_archive(str(KAGGLE_WORKING/'phase1_round3_results_bundle'),'zip',RESULTS)
print('Results CSV:',RESULTS/'round3.csv')
print('Winner:',RESULTS/'round3_resolved_winner.json')
print('Downloadable archive:',archive)
print('Detailed resumable artifacts:',EXPERIMENTS)
disk_status()
